<a href="https://colab.research.google.com/github/CarenMoreno/Actividad3-PCA-ExpresionGenica-IMC/blob/main/multiplex_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multiplex IF pipeline — PhenoImager / inForm component data

Single-cell analysis of Opal multiplex IF (Akoya PhenoImager Fusion), done in Python from the **unmixed
component images exported by inForm**. inForm is used only for spectral unmixing (it needs the Opal spectral
library); segmentation, feature extraction, positivity calls and phenotyping are done here, following the
strategy of Eng et al. 2022 (*A framework for multiplex imaging optimization and reproducible analysis*,
bioRxiv 10.1101/2021.11.29.470281; github.com/engjen/cycIF_Validation).

| Section | What it does | Output |
|---|---|---|
| 1 | Configuration: paths, panel, parameters | — |
| 2 | Inventory of ROIs and sample metadata | `inventory.csv`, `samples.csv` |
| 3 | Image QC: per-channel signal coverage, visual check | `qc_channels.csv` |
| 4 | Audit of inForm phenotype calls against their own channel | `qc_inform_audit.csv` |
| 5 | Tissue mask and nuclear/cell segmentation (StarDist or Cellpose) | `masks/*.tif` |
| 6 | Segmentation agreement with inForm | `qc_segmentation_vs_inform.csv` |
| 7 | Per-cell feature extraction | `cells_features.parquet` |
| 8 | Positivity thresholds, signal-to-background ratio (SBR), agreement with inForm | `thresholds.csv`, `qc_sbr.csv` |
| 9 | Phenotypes, densities per ROI / slide, whole-section maps | `cells_phenotyped.parquet`, `densities_*.csv` |
| 10–11 | Batch normalization, clustering, spatial analysis | *to build* |

**Input per ROI or field** (inForm export, one folder per slide or patient):
`<slide>_[x,y]_component_data.tif` (required) and, optionally for comparison,
`_binary_seg_maps.tif` and `_cell_seg_data.txt`.

**Whole-tissue mode.** The same code handles hand-picked ROIs and a full grid of fields covering the tissue
(Phenochart ROI tool → inForm batch). With a grid, each field "owns" the part of the slide closer to its
centre than to any neighbour's: cells and tissue area in overlap zones are counted once. Fields that are
almost all glass are skipped. Masks and per-field features are cached in `output/`, so an interrupted
run resumes where it stopped.

## 0. Environment (Google Colab)
1. Download the inForm file from the link you were given (`..._component_data.tif`, or a `.zip` that contains it).
2. In Colab, click the **folder icon** on the left bar and drag the file into the panel (or use the upload button). It is saved in `/content`. Wait until the upload finishes.
3. Run the next cell: it installs the two packages Colab lacks (StarDist, imagecodecs), unzips `.zip` files and checks that the image is there.

Files uploaded this way are deleted when the Colab session ends — download the results at the end of the notebook.

## 1. Configuration

In [ ]:
!pip install -q stardist imagecodecs

In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "stardist", "imagecodecs", "scanpy", "igraph", "leidenalg"], check=True)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

PROJECT_DIR = Path("/content")   # the .tif uploaded in the left panel lives here

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import filters, morphology, measure, segmentation
from sklearn.metrics import roc_auc_score, cohen_kappa_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50, "display.width", 200)

In [ ]:
# ---- Paths ----
# Folder that contains the inForm exports. It is searched recursively, so it can be one patient folder
# (e.g. r"D:\Endometrial Cancer\Panel 1 Akoya\2838") or the parent folder with all patients.
DATA_DIR = Path("/content")
OUT_DIR = Path("/content/output")
META_DIR = Path("/content/metadata")
MODEL_DIR = Path("/content/models")
for d in (OUT_DIR, OUT_DIR / "masks", META_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)
# ---- Panel 1 ----
# component: channel name exactly as stored in the component_data.tif.
# compartment: where the marker is read for positivity — "nucleus", "membrane" (ring around the nucleus)
# or "cell" (nucleus + ring; used for cytoplasmic granules such as GzmB).
PANEL = pd.DataFrame([
    ("DAPI",  "DAPI",             "nucleus"),
    ("PD1",   "PD1 (Opal 480)",   "membrane"),
    ("CD3",   "CD3 (Opal 520)",   "membrane"),
    ("Ki67",  "Ki67 (Opal 570)",  "nucleus"),
    ("CD8",   "CD8 (Opal 620)",   "membrane"),
    ("Foxp3", "Foxp3 (Opal 690)", "nucleus"),
    ("GzmB",  "GzmB (Opal 780)",  "cell"),
    ("AF",    "Autofluorescence", "cell"),
], columns=["marker", "component", "compartment"])
MARKERS = [m for m in PANEL["marker"] if m not in ("DAPI", "AF")]

# ---- Segmentation ----
SEG_BACKEND = "stardist"      # "stardist" (2D_versatile_fluo) or "cellpose" (nuclei model)
STARDIST_SCALE = 1.25         # upscaling before StarDist; small, packed nuclei at 0.5 µm/px are under-split at 1.0
CELL_EXPANSION_UM = 1.5       # nucleus -> cell expansion; paper: 5 px at 0.325 µm/px ≈ 1.6 µm
CELLPOSE_DIAMETER_UM = 8.0    # only for cellpose: expected nuclear diameter
MIN_NUCLEUS_UM2 = 7.0         # remove smaller objects (debris)
TISSUE_REL_THRESHOLD = 0.2    # tissue mask: smoothed DAPI+AF above 20% of its 95th percentile (glass ≈ 0)
TISSUE_MIN_HOLE_UM2 = 2000    # tissue mask: fill holes smaller than this

# ---- QC ----
QC_EMPTY_FRAC = 0.0005        # channel flagged if < 0.05% of pixels have signal (median over ROIs)
QC_DIFFUSE_FRAC = 0.30        # channel flagged if > 30% of pixels have signal

# ---- Whole-tissue grids ----
MIN_TISSUE_FRAC = 0.05        # skip fields with < 5% tissue (glass at the section border)

## 2. Inventory and sample metadata

In [ ]:
ROI_REGEX = re.compile(r"^(?P<slide>.+?)_\[(?P<x>\d+),(?P<y>\d+)\]_component_data.*\.tif$")


def build_inventory(data_dir):
    rows = []
    for f in sorted(data_dir.rglob("*component_data*.tif")):
        m = ROI_REGEX.match(f.name)
        if m is None:
            print("Skipped (name not recognised):", f.name)
            continue
        stem = f.name[: f.name.index("_component_data")]
        rows.append({
            "slide": m["slide"], "roi": f"{m['x']}_{m['y']}", "roi_id": stem,
            "center_x_um": float(m["x"]), "center_y_um": float(m["y"]),
            "component": f,
            "inform_seg": f.with_name(stem + "_binary_seg_maps.tif"),
            "inform_cells": f.with_name(stem + "_cell_seg_data.txt"),
        })
    if not rows:
        print("Files in", data_dir, ":")
        for p in sorted(data_dir.iterdir()):
            print("  ", p.name)
        raise FileNotFoundError("No inForm '_component_data.tif' file found (see the list above).")
    inv = pd.DataFrame(rows)
    inv["has_inform"] = inv["inform_seg"].map(Path.exists) & inv["inform_cells"].map(Path.exists)
    geo = [field_geometry(f) for f in inv["component"]]
    inv["px_um"] = [g[0] for g in geo]
    inv["w_um"] = [g[2] * g[0] for g in geo]
    inv["h_um"] = [g[1] * g[0] for g in geo]
    inv["x0_um"] = inv["center_x_um"] - inv["w_um"] / 2
    inv["y0_um"] = inv["center_y_um"] - inv["h_um"] / 2
    # neighbours: fields of the same slide whose extent overlaps this one
    inv["neighbours"] = [
        [j for j in inv.index[(inv["slide"] == r["slide"]) & (inv.index != i)
                              & (abs(inv["center_x_um"] - r["center_x_um"]) < (inv["w_um"] + r["w_um"]) / 2)
                              & (abs(inv["center_y_um"] - r["center_y_um"]) < (inv["h_um"] + r["h_um"]) / 2)]]
        for i, r in inv.iterrows()]
    return inv


def field_geometry(path):
    """(µm per pixel, height px, width px) from the TIFF header, without reading pixels."""
    with tifffile.TiffFile(path) as tf:
        p = tf.pages[0]
        num, den = p.tags["XResolution"].value
        unit = p.tags["ResolutionUnit"].value
        return (1e4 if unit == 3 else 25400.0) * den / num, p.shape[0], p.shape[1]


inventory = build_inventory(DATA_DIR)
print(f"{len(inventory)} ROIs/fields in {inventory['slide'].nunique()} slides; "
      f"{inventory['has_inform'].sum()} with inForm segmentation for comparison")
display(inventory.groupby("slide").agg(n_fields=("roi", "size"),
                                       overlapping_fields=("neighbours", lambda x: sum(len(v) > 0 for v in x))))

`samples.csv` links each slide to patient-level information. It is created with empty columns the first time;
fill it in (patient, group, staining batch) and re-run. The staining batch is what batch normalization uses later.

In [ ]:
samples_path = META_DIR / "samples.csv"
if samples_path.exists():
    samples = pd.read_csv(samples_path, dtype=str)
    new = sorted(set(inventory["slide"]) - set(samples["slide"]))
    if new:
        print("Slides missing from samples.csv:", new)
else:
    samples = pd.DataFrame({"slide": sorted(inventory["slide"].unique())})
    for col in ("patient", "group", "staining_batch", "notes"):
        samples[col] = ""
    samples.to_csv(samples_path, index=False)
    print(f"Template written: {samples_path} — fill it in")
display(samples)

## 3. Image QC
Reads the component TIFF (float32, one page per unmixed channel; names and pixel size come from the
file metadata).

In [ ]:
def load_component(path):
    """Return (image [C, H, W] float32, channel names, µm per pixel)."""
    with tifffile.TiffFile(path) as tf:
        pages = [p for p in tf.pages if "<ImageType>FullResolution</ImageType>" in (p.description or "")]
        names = [re.search(r"<Name>(.*?)</Name>", p.description).group(1) for p in pages]
        img = np.stack([p.asarray() for p in pages]).astype(np.float32)
        num, den = pages[0].tags["XResolution"].value
        unit = pages[0].tags["ResolutionUnit"].value       # 3 = centimetre
        um_per_px = (1e4 if unit == 3 else 25400.0) * den / num
    return img, names, um_per_px


def channel(img, names, marker):
    comp = PANEL.set_index("marker").loc[marker, "component"]
    return img[names.index(comp)]


img0, names0, PX_UM = load_component(inventory["component"].iloc[0])
missing = set(PANEL["component"]) - set(names0)
print(f"Channels: {names0}\nPixel size: {PX_UM:.4f} µm; ROI size: {img0.shape[2]*PX_UM:.0f} x {img0.shape[1]*PX_UM:.0f} µm")
print("Panel OK" if not missing else f"WARNING: channels in PANEL not found in file: {missing}")

**Channel coverage.** Fraction of pixels with signal > 0 and the 99th/99.9th percentiles per channel and ROI.
Unmixed Opal channels should be zero almost everywhere except at positive cells. A channel that is almost
empty (failed staining or unmixing), or one with signal in most pixels (diffuse background / unmixing
residue), makes the positivity calls of that marker unreliable.

In [ ]:
def channel_qc(inv):
    rows = []
    for _, r in inv.iterrows():
        img, names, _ = load_component(r["component"])
        for m in PANEL["marker"]:
            c = channel(img, names, m)
            rows.append({"slide": r["slide"], "roi": r["roi"], "marker": m,
                         "frac_pos": float((c > 0).mean()),
                         "p99": float(np.percentile(c, 99)), "p999": float(np.percentile(c, 99.9)),
                         "max": float(c.max())})
    return pd.DataFrame(rows)


qc_ch = channel_qc(inventory)
qc_ch.to_csv(OUT_DIR / "qc_channels.csv", index=False)
display(qc_ch.pivot_table(index=["slide", "roi"], columns="marker", values="frac_pos").round(3))

# Flag markers whose channel is almost empty or has signal almost everywhere (median over ROIs).
cov = qc_ch[qc_ch["marker"].isin(MARKERS)].groupby("marker")["frac_pos"].median()
CHANNEL_FLAGS = {m: ("near-empty channel" if v < QC_EMPTY_FRAC else "diffuse background")
                 for m, v in cov.items() if v < QC_EMPTY_FRAC or v > QC_DIFFUSE_FRAC}
for m, why in CHANNEL_FLAGS.items():
    print(f"WARNING {m}: {why} (median fraction of pixels > 0 = {cov[m]:.4f}); its calls need visual review")

In [ ]:
def show_roi(img, names, markers=None, crop=None, ncol=4):
    markers = markers or list(PANEL["marker"])
    ys, xs = (slice(None), slice(None)) if crop is None else crop
    nrow = int(np.ceil(len(markers) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.2 * nrow), squeeze=False)
    for ax, m in zip(axes.flat, markers):
        c = channel(img, names, m)[ys, xs]
        hi = np.percentile(c, 99.8) or c.max() or 1
        ax.imshow(c, cmap="magma", vmin=0, vmax=hi)
        ax.set_title(f"{m} (0–{hi:.2f})", fontsize=9)
        ax.axis("off")
    for ax in list(axes.flat)[len(markers):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


r0 = inventory.iloc[0]
print(r0["roi_id"])
show_roi(img0, names0)

## 4. Audit of inForm phenotype calls
For each ROI with inForm output: using inForm's own cell masks, how well does each channel separate the
cells inForm called positive from the negative ones (ROC AUC)? A trustworthy call has AUC ≈ 1 for its own
channel. A low AUC on the diagonal means inForm's classifier for that marker is not driven by that marker's
signal (e.g. trained on autofluorescence or morphology).

In [ ]:
def compartment_masks(nuc, cell):
    ring = np.where(nuc > 0, 0, cell)
    return {"nucleus": nuc, "membrane": ring, "cell": cell}


def mean_intensities(masks, img, names):
    """Mean of every channel in every compartment -> DataFrame indexed by label."""
    stack = np.moveaxis(img, 0, -1)
    out = []
    for comp, lab in masks.items():
        t = pd.DataFrame(measure.regionprops_table(lab, stack, properties=("label", "intensity_mean")))
        t.columns = ["label"] + [f"{comp}_{n}" for n in names]
        out.append(t.set_index("label"))
    return pd.concat(out, axis=1)


def read_inform_cells(path):
    return pd.read_csv(path, sep="\t", low_memory=False)


def inform_audit(inv):
    rows = []
    for _, r in inv[inv["has_inform"]].iterrows():
        img, names, _ = load_component(r["component"])
        seg = tifffile.imread(r["inform_seg"]).astype(np.int32)
        nuc, mem = seg[1], seg[2]
        cell = np.where(nuc > 0, nuc, mem)
        feats = mean_intensities(compartment_masks(nuc, cell), img, names)
        cells = read_inform_cells(r["inform_cells"]).set_index("Cell ID").join(feats, how="inner")
        p = PANEL.set_index("marker")
        for m in MARKERS:
            y = cells[f"Phenotype-{m}"].eq(f"{m}+").astype(int)
            rec = {"slide": r["slide"], "roi": r["roi"], "phenotype": m, "n_pos": int(y.sum())}
            for m2 in PANEL["marker"]:
                col = f"{p.loc[m, 'compartment']}_{p.loc[m2, 'component']}"
                rec[m2] = roc_auc_score(y, cells[col]) if 0 < y.sum() < len(y) else np.nan
            rows.append(rec)
    return pd.DataFrame(rows)


audit = inform_audit(inventory)
if audit.empty:
    own = pd.DataFrame(columns=["slide", "roi", "phenotype", "own_auc"])
    print("No inForm segmentation exported — audit skipped")
else:
    audit.to_csv(OUT_DIR / "qc_inform_audit.csv", index=False)
    own = audit.assign(own_auc=[r[r["phenotype"]] for _, r in audit.iterrows()])
    display(own.pivot_table(index=["slide", "roi"], columns="phenotype", values="own_auc").round(2)
            .style.background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1))
    print("Mean AUC of each inForm call (rows) against each channel (columns):")
    display(audit.groupby("phenotype")[list(PANEL["marker"])].mean().round(2))

## 5. Tissue mask and segmentation
- **Tissue mask**: smoothed DAPI + autofluorescence above a fraction of its 95th percentile (glass is ≈ 0), holes filled. Used for areas and densities.
- **Nuclei**: StarDist `2D_versatile_fluo` (default; image upscaled by `STARDIST_SCALE`) or Cellpose `nuclei`
  on the DAPI channel.
- **Cells**: nuclei expanded by `CELL_EXPANSION_UM` without overlap (as in the paper, since there is no
  pan-membrane marker in this panel). Membrane compartment = cell − nucleus.

In [ ]:
def tissue_mask(img, names, px_um):
    base = channel(img, names, "DAPI") + channel(img, names, "AF")
    sm = filters.gaussian(base, sigma=10 / px_um)
    mask = sm > TISSUE_REL_THRESHOLD * np.percentile(sm, 95)
    mask = morphology.remove_small_holes(mask, int(TISSUE_MIN_HOLE_UM2 / px_um ** 2))
    return morphology.remove_small_objects(mask, int(TISSUE_MIN_HOLE_UM2 / px_um ** 2))


_SEG_MODEL = None
STARDIST_URL = "https://github.com/stardist/stardist-models/releases/download/v0.1/python_2D_versatile_fluo.zip"


def load_stardist(model_dir=Path.cwd() / "models"):
    """Load StarDist '2D_versatile_fluo' from a local folder, downloading it once if needed.
    Avoids StarDist2D.from_pretrained, whose Keras download creates a symlink that fails on
    Windows without administrator rights (WinError 1314)."""
    import shutil, urllib.request, zipfile
    from stardist.models import StarDist2D
    mdir = model_dir / "2D_versatile_fluo"
    files = ("config.json", "thresholds.json", "weights_best.h5")
    if not all((mdir / f).exists() for f in files):
        mdir.mkdir(parents=True, exist_ok=True)
        cache = Path.home() / ".keras/models/StarDist2D/2D_versatile_fluo/2D_versatile_fluo_extracted"
        if all((cache / f).exists() for f in files):
            for f in files:
                shutil.copy(cache / f, mdir / f)
        else:
            zpath = model_dir / "2D_versatile_fluo.zip"
            urllib.request.urlretrieve(STARDIST_URL, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(mdir)
    return StarDist2D(None, name="2D_versatile_fluo", basedir=str(model_dir))


def segment_nuclei(dapi, px_um):
    global _SEG_MODEL
    if SEG_BACKEND == "stardist":
        from csbdeep.utils import normalize
        if _SEG_MODEL is None:
            _SEG_MODEL = load_stardist()
        tiles = tuple(max(1, s // 1024) for s in dapi.shape)
        lab, _ = _SEG_MODEL.predict_instances(normalize(dapi, 1, 99.8), scale=STARDIST_SCALE, n_tiles=tiles,
                                              show_tile_progress=False, verbose=False)
    elif SEG_BACKEND == "cellpose":
        from cellpose import models
        if _SEG_MODEL is None:
            _SEG_MODEL = models.Cellpose(gpu=False, model_type="nuclei")
        lab, *_ = _SEG_MODEL.eval(dapi, diameter=CELLPOSE_DIAMETER_UM / px_um, channels=[0, 0])
    else:
        raise ValueError(SEG_BACKEND)
    lab = morphology.remove_small_objects(lab.astype(np.int32), int(MIN_NUCLEUS_UM2 / px_um ** 2))
    return segmentation.relabel_sequential(lab)[0]


def segment_roi(r, overwrite=False):
    out = OUT_DIR / "masks" / f"{r['roi_id']}_masks.tif"
    if out.exists() and not overwrite:
        m = tifffile.imread(out)
        return m[0].astype(bool), m[1].astype(np.int32), m[2].astype(np.int32)
    img, names, px = load_component(r["component"])
    tissue = tissue_mask(img, names, px)
    nuc = segment_nuclei(channel(img, names, "DAPI"), px)
    nuc[~tissue] = 0
    nuc = segmentation.relabel_sequential(nuc)[0]
    cell = segmentation.expand_labels(nuc, distance=round(CELL_EXPANSION_UM / px))
    tifffile.imwrite(out, np.stack([tissue.astype(np.int32), nuc, cell]), compression="zlib")
    return tissue, nuc, cell


import time
t0 = time.time()
tissue_frac = {}
for i, r in inventory.iterrows():
    tissue, nuc, _ = segment_roi(r)
    tissue_frac[i] = tissue.mean()
    if (i + 1) % 10 == 0 or i + 1 == len(inventory):
        el = time.time() - t0
        print(f"[{i + 1}/{len(inventory)}] {el / 60:.1f} min elapsed, "
              f"~{el / (i + 1) * (len(inventory) - i - 1) / 60:.0f} min left")
inventory["tissue_frac"] = pd.Series(tissue_frac)
inventory["use"] = inventory["tissue_frac"] >= MIN_TISSUE_FRAC
print(f"{(~inventory['use']).sum()} fields skipped (< {MIN_TISSUE_FRAC:.0%} tissue)")

In [ ]:
def show_segmentation(r, crop=(slice(400, 700), slice(600, 1000))):
    img, names, _ = load_component(r["component"])
    tissue, nuc, cell = segment_roi(r)
    dapi = channel(img, names, "DAPI")[crop]
    panels = [("Python (nuclei + cell)", nuc[crop], cell[crop])]
    if r["has_inform"]:
        s = tifffile.imread(r["inform_seg"]).astype(np.int32)
        panels.append(("inForm", s[1][crop], np.where(s[1] > 0, s[1], s[2])[crop]))
    fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 5), squeeze=False)
    for ax, (title, n, c) in zip(axes.flat, panels):
        ax.imshow(dapi, cmap="gray", vmax=np.percentile(dapi, 99.5))
        ax.contour(segmentation.find_boundaries(n), [0.5], colors="cyan", linewidths=0.5)
        ax.contour(segmentation.find_boundaries(c), [0.5], colors="yellow", linewidths=0.5)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_segmentation(inventory.loc[inventory["tissue_frac"].idxmax()])

## 6. Segmentation agreement with inForm
Nuclei matched by IoU ≥ 0.5. Precision = fraction of Python nuclei that match an inForm nucleus;
recall = fraction of inForm nuclei recovered. Disagreement is not automatically a Python error — check the
overlay above.

In [ ]:
def match_nuclei(gt, pred, thr=0.5):
    """Greedy one-to-one IoU matching; returns (n_matched, n_gt, n_pred)."""
    gt_ids, gt_area = np.unique(gt[gt > 0], return_counts=True)
    pr_ids, pr_area = np.unique(pred[pred > 0], return_counts=True)
    both = (gt > 0) & (pred > 0)
    pairs, inter = np.unique(np.stack([gt[both], pred[both]]), axis=1, return_counts=True)
    ga = dict(zip(gt_ids, gt_area))
    pa = dict(zip(pr_ids, pr_area))
    iou = inter / (np.array([ga[g] for g in pairs[0]]) + np.array([pa[p] for p in pairs[1]]) - inter)
    order = np.argsort(-iou)
    used_g, used_p, n = set(), set(), 0
    for k in order:
        if iou[k] < thr:
            break
        g, p = pairs[0, k], pairs[1, k]
        if g not in used_g and p not in used_p:
            used_g.add(g)
            used_p.add(p)
            n += 1
    return n, len(gt_ids), len(pr_ids)


rows = []
for _, r in inventory[inventory["has_inform"]].iterrows():
    tissue, nuc, cell = segment_roi(r)
    inf = tifffile.imread(r["inform_seg"]).astype(np.int32)[1]
    n, ng, npred = match_nuclei(inf, nuc)
    rows.append({"slide": r["slide"], "roi": r["roi"], "n_inform": ng, "n_python": npred,
                 "precision": n / npred, "recall": n / ng, "f1": 2 * n / (ng + npred)})
seg_qc = pd.DataFrame(rows)
seg_qc.to_csv(OUT_DIR / "qc_segmentation_vs_inform.csv", index=False)
display(seg_qc.round(3))

## 7. Feature extraction
One row per cell: ROI, position (pixels and slide µm, same frame as inForm's `Cell X/Y Position`),
morphology, and the mean of every channel in every compartment. `sig_<marker>` is the mean in the
marker's compartment from `PANEL` and is what positivity is called on.

Overlapping fields: a cell is kept only in the field whose centre is closest to the cell's centroid, and
tissue area is counted only in that same "owned" region, so nothing is counted twice.

In [ ]:
def ownership_mask(r, inv, shape):
    """True where this field is the closest field centre (Voronoi split of overlap zones)."""
    h, w = shape
    xs = r["x0_um"] + (np.arange(w) + 0.5) * r["px_um"]
    ys = r["y0_um"] + (np.arange(h) + 0.5) * r["px_um"]
    own = np.ones(shape, bool)
    d_self = (xs[None, :] - r["center_x_um"]) ** 2 + (ys[:, None] - r["center_y_um"]) ** 2
    for j in r["neighbours"]:
        nb = inv.loc[j]
        d_nb = (xs[None, :] - nb["center_x_um"]) ** 2 + (ys[:, None] - nb["center_y_um"]) ** 2
        own &= d_self <= d_nb
    return own


def extract_features(r):
    img, names, px = load_component(r["component"])
    tissue, nuc, cell = segment_roi(r)
    own = ownership_mask(r, inventory, nuc.shape)
    masks = compartment_masks(nuc, cell)
    props = pd.DataFrame(measure.regionprops_table(
        nuc, properties=("label", "centroid", "area", "eccentricity", "solidity"))).set_index("label")
    props.columns = ["y_px", "x_px", "nucleus_area_um2", "nucleus_eccentricity", "nucleus_solidity"]
    props["nucleus_area_um2"] *= px ** 2
    props["cell_area_um2"] = pd.Series(np.bincount(cell.ravel())[1:] * px ** 2,
                                       index=np.arange(1, cell.max() + 1))
    df = props.join(mean_intensities(masks, img, names))
    h, w = nuc.shape
    df["x_um"] = r["center_x_um"] - w * px / 2 + df["x_px"] * px
    df["y_um"] = r["center_y_um"] - h * px / 2 + df["y_px"] * px
    for _, p in PANEL.iterrows():
        df[f"sig_{p['marker']}"] = df[f"{p['compartment']}_{p['component']}"]
    df = df[own[df["y_px"].round().astype(int).clip(0, h - 1), df["x_px"].round().astype(int).clip(0, w - 1)]]
    df.insert(0, "roi", r["roi"])
    df.insert(0, "slide", r["slide"])
    df["tissue_area_mm2"] = (tissue & own).sum() * px ** 2 / 1e6
    num = df.select_dtypes("float64").columns
    df[num] = df[num].astype(np.float32)
    df[["x_um", "y_um"]] = df[["x_um", "y_um"]].astype(np.float64)
    return df.reset_index().rename(columns={"label": "cell_id"})


def features_cached(r, overwrite=False):
    out = OUT_DIR / "features" / f"{r['roi_id']}.parquet"
    if out.exists() and not overwrite:
        return pd.read_parquet(out)
    out.parent.mkdir(exist_ok=True)
    df = extract_features(r)
    df.to_parquet(out)
    return df


cells = pd.concat([features_cached(r) for _, r in inventory[inventory["use"]].iterrows()], ignore_index=True)
cells.to_parquet(OUT_DIR / "cells_features.parquet")
print(f"{len(cells):,} cells")
per_field = cells.groupby(["slide", "roi"]).agg(n_cells=("cell_id", "size"),
                                                tissue_mm2=("tissue_area_mm2", "first"),
                                                nucleus_um2=("nucleus_area_um2", "median"))
display(per_field.groupby("slide").agg(n_fields=("n_cells", "size"), n_cells=("n_cells", "sum"),
                                       tissue_mm2=("tissue_mm2", "sum"),
                                       median_nucleus_um2=("nucleus_um2", "median")).round(2))

## 8. Positivity thresholds and SBR
Thresholds are set **per marker** on `sig_<marker>` and stored in `metadata/thresholds.csv`, which is
created on the first run with:
- `threshold`: proposal by Otsu on log1p(signal) over all cells — the value actually used;
- `inform_matched`: for reference only, the threshold that best reproduces inForm's calls (max Cohen's κ),
  filled in only for markers whose inForm calls passed the audit in Section 4 (own-channel AUC ≥ 0.9).

As in the paper, the final threshold is a visual decision: review the histograms and overlays below, edit
`threshold` (and set `source = manual`), and re-run from here. Use `slide = *` for one threshold for all
slides, or add per-slide rows.

SBR per marker and ROI as in the paper: mean signal of positive cells / mean signal of negative cells.
In unmixed component data the negative background is close to 0, so SBR becomes very large for clean
channels; compare it between ROIs/slides of the same marker rather than as an absolute number.

In [ ]:
def propose_threshold(x):
    x = np.log1p(np.clip(x[np.isfinite(x)], 0, None))
    return float(np.expm1(filters.threshold_otsu(x)))


def link_to_inform(r, df):
    """Python cells of one ROI linked to the inForm cell whose nucleus contains their centroid."""
    inf_nuc = tifffile.imread(r["inform_seg"]).astype(np.int32)[1]
    sub = df[(df["slide"] == r["slide"]) & (df["roi"] == r["roi"])].copy()
    yy = sub["y_px"].round().astype(int).clip(0, inf_nuc.shape[0] - 1)
    xx = sub["x_px"].round().astype(int).clip(0, inf_nuc.shape[1] - 1)
    sub["inform_id"] = inf_nuc[yy, xx]
    inf = read_inform_cells(r["inform_cells"]).set_index("Cell ID")
    sub = sub[sub["inform_id"] > 0].drop_duplicates("inform_id", keep=False)
    return sub.join(inf[[f"Phenotype-{m}" for m in MARKERS]], on="inform_id")


inform_ok = inventory["has_inform"].any()
linked = (pd.concat([link_to_inform(r, cells) for _, r in inventory[inventory["has_inform"]].iterrows()])
          if inform_ok else None)
own_auc = own.groupby("phenotype")["own_auc"].mean() if inform_ok else pd.Series(dtype=float)


def inform_matched_threshold(m):
    if not inform_ok or own_auc.get(m, 0) < 0.9:
        return np.nan
    y, v = linked[f"Phenotype-{m}"].eq(f"{m}+"), linked[f"sig_{m}"]
    grid = np.unique(np.quantile(cells[f"sig_{m}"], np.linspace(0.5, 0.9999, 400)))
    return float(max(grid, key=lambda t: cohen_kappa_score(y, v > t)))


thr_path = META_DIR / "thresholds.csv"
if thr_path.exists():
    thresholds = pd.read_csv(thr_path, dtype={"slide": str})
else:
    thresholds = pd.DataFrame({"slide": "*", "marker": MARKERS,
                               "threshold": [propose_threshold(cells[f"sig_{m}"].values) for m in MARKERS],
                               "inform_matched": [inform_matched_threshold(m) for m in MARKERS],
                               "source": "auto_otsu",
                               "qc_flag": [CHANNEL_FLAGS.get(m, "") for m in MARKERS]})
    thresholds.to_csv(thr_path, index=False)
    print(f"Proposed thresholds written to {thr_path} — review and edit")
display(thresholds)


def get_threshold(marker, slide):
    t = thresholds[(thresholds["marker"] == marker) & (thresholds["slide"] == slide)]
    if t.empty:
        t = thresholds[(thresholds["marker"] == marker) & (thresholds["slide"] == "*")]
    return float(t["threshold"].iloc[0])


for m in MARKERS:
    cells[f"{m}_pos"] = [v > get_threshold(m, s) for v, s in zip(cells[f"sig_{m}"], cells["slide"])]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, m in zip(axes.flat, MARKERS):
    v = np.log1p(cells[f"sig_{m}"].clip(lower=0))
    ax.hist(v, bins=150, color="0.4")
    ax.set_yscale("log")
    ax.axvline(np.log1p(get_threshold(m, "*")), color="crimson", label="threshold")
    ref = thresholds.loc[(thresholds["marker"] == m) & (thresholds["slide"] == "*"), "inform_matched"]
    if len(ref) and pd.notna(ref.iloc[0]):
        ax.axvline(np.log1p(ref.iloc[0]), color="steelblue", ls="--", label="inForm-matched")
        ax.legend(fontsize=8)
    flag = f" [{CHANNEL_FLAGS[m]}]" if m in CHANNEL_FLAGS else ""
    ax.set_title(f"{m}: {cells[f'{m}_pos'].mean():.1%} positive{flag}", fontsize=10)
    ax.set_xlabel("log1p(mean signal)")
plt.tight_layout()
plt.show()

In [ ]:
def show_calls(r, marker, crop=None):
    """Marker channel with the positive cells outlined (green) — visual check of a threshold."""
    img, names, _ = load_component(r["component"])
    _, nuc, cell = segment_roi(r)
    c = channel(img, names, marker)
    sub = cells[(cells["slide"] == r["slide"]) & (cells["roi"] == r["roi"])]
    pos = np.isin(cell, sub.loc[sub[f"{marker}_pos"], "cell_id"].values)
    if crop is not None:
        c, pos = c[crop], pos[crop]
    fig, ax = plt.subplots(figsize=(10, 7.5))
    ax.imshow(c, cmap="gray", vmax=np.percentile(c, 99.8) or c.max())
    ax.contour(pos, [0.5], colors="lime", linewidths=0.6)
    ax.set_title(f"{r['roi_id']} — {marker}+ cells")
    ax.axis("off")
    plt.show()


show_calls(inventory.iloc[0], "CD3")

In [ ]:
sbr = []
for (s, roi), g in cells.groupby(["slide", "roi"]):
    for m in MARKERS:
        pos, neg = g.loc[g[f"{m}_pos"], f"sig_{m}"], g.loc[~g[f"{m}_pos"], f"sig_{m}"]
        sbr.append({"slide": s, "roi": roi, "marker": m, "n_pos": len(pos),
                    "pct_pos": len(pos) / len(g) * 100,
                    "mean_pos": pos.mean(), "mean_neg": neg.mean(),
                    "SBR": pos.mean() / neg.mean() if len(pos) and neg.mean() > 0 else np.nan})
sbr = pd.DataFrame(sbr)
sbr.to_csv(OUT_DIR / "qc_sbr.csv", index=False)
print("Median over fields of % positive and SBR:")
display(sbr.groupby(["slide", "marker"])[["pct_pos", "SBR"]].median().unstack("marker").round(2))

**Agreement with inForm calls.** Each Python cell is linked to the inForm cell whose nucleus contains its
centroid. Cohen's κ per marker; only meaningful for markers whose inForm calls passed the audit in Section 4.

In [ ]:
if inform_ok:
    lk = linked.join(cells[[f"{m}_pos" for m in MARKERS]])
    agree = []
    for m in MARKERS:
        y_inf = lk[f"Phenotype-{m}"].eq(f"{m}+")
        agree.append({"marker": m, "n_linked_cells": len(lk),
                      "inform_pct_pos": 100 * y_inf.mean(), "python_pct_pos": 100 * lk[f"{m}_pos"].mean(),
                      "kappa": cohen_kappa_score(y_inf, lk[f"{m}_pos"]),
                      "inform_own_channel_AUC": own_auc.get(m, np.nan),
                      "qc_flag": CHANNEL_FLAGS.get(m, "")})
    display(pd.DataFrame(agree).round(3))

## 9. Phenotypes and densities
Rule-based phenotypes from the single-marker calls. Densities are cells per mm² of tissue (tissue mask
from Section 5).

In [ ]:
PHENOTYPES = {
    "T cell (CD3+)":            lambda d: d["CD3_pos"],
    "CD8 T (CD3+CD8+)":         lambda d: d["CD3_pos"] & d["CD8_pos"],
    "CD8- T (CD3+CD8-)":        lambda d: d["CD3_pos"] & ~d["CD8_pos"],
    "Treg (CD3+CD8-Foxp3+)":    lambda d: d["CD3_pos"] & ~d["CD8_pos"] & d["Foxp3_pos"],
    "Cytotoxic (CD8+GzmB+)":    lambda d: d["CD8_pos"] & d["GzmB_pos"],
    "PD1+ CD8 T":               lambda d: d["CD3_pos"] & d["CD8_pos"] & d["PD1_pos"],
    "Proliferating (Ki67+)":    lambda d: d["Ki67_pos"],
    "Proliferating CD8 T":      lambda d: d["CD3_pos"] & d["CD8_pos"] & d["Ki67_pos"],
}
for name, rule in PHENOTYPES.items():
    cells[name] = rule(cells)
    used = [m for m in CHANNEL_FLAGS if m in name]
    if used:
        print(f"WARNING '{name}' uses flagged marker(s) {used}: interpret with caution")
cells.to_parquet(OUT_DIR / "cells_phenotyped.parquet")

agg = {"n_cells": ("cell_id", "size"), "tissue_mm2": ("tissue_area_mm2", "first")}
agg.update({name: (name, "sum") for name in PHENOTYPES})
roi_counts = cells.groupby(["slide", "roi"]).agg(**agg).reset_index()
dens = roi_counts.copy()
for name in PHENOTYPES:
    dens[name] = roi_counts[name] / roi_counts["tissue_mm2"]
dens.to_csv(OUT_DIR / "densities_per_roi.csv", index=False)

slide_counts = roi_counts.groupby("slide")[["n_cells", "tissue_mm2", *PHENOTYPES]].sum()
slide_dens = slide_counts[list(PHENOTYPES)].div(slide_counts["tissue_mm2"], axis=0)
slide_dens.insert(0, "tissue_mm2", slide_counts["tissue_mm2"])
slide_dens.to_csv(OUT_DIR / "densities_per_slide.csv")
print("Cells / mm² tissue, per slide (pooled over ROIs):")
display(slide_dens.round(1).T)

**Whole-section maps.** Local density of a phenotype on a hexagonal grid (150 µm) in slide coordinates —
most useful in whole-tissue mode. Only hexagons containing cells are drawn, so the shape is the tissue;
hexagons at the tissue edge are partly empty and underestimate density.

In [ ]:
def plot_density_map(slide, phenotype, hex_um=150, ax=None):
    d = cells[cells["slide"] == slide]
    ax = ax or plt.subplots(figsize=(7, 9))[1]
    x0, x1, y0, y1 = d["x_um"].min(), d["x_um"].max(), d["y_um"].min(), d["y_um"].max()
    gx = max(1, int((x1 - x0) / hex_um))
    gy = max(1, int((y1 - y0) / (hex_um * np.sqrt(3))))
    hex_area_mm2 = ((x1 - x0) / gx) * ((y1 - y0) / gy) / 2 / 1e6   # matplotlib hexbin cell area
    hb = ax.hexbin(d["x_um"], d["y_um"], C=d[phenotype].astype(float),
                   reduce_C_function=lambda v: np.sum(v) / hex_area_mm2,
                   gridsize=(gx, gy), extent=(x0, x1, y0, y1), mincnt=1, cmap="magma_r", linewidths=0)
    vals = hb.get_array()
    hb.set_clim(0, np.percentile(vals, 99) if vals.max() > 0 else 1)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_title(f"{slide}\n{phenotype}", fontsize=10)
    ax.set_xlabel("x (µm)")
    plt.colorbar(hb, ax=ax, label="cells / mm² (approx., per hexagon)", shrink=0.6)
    return ax


for slide in cells["slide"].unique():
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    for ax, ph in zip(axes, ["CD8 T (CD3+CD8+)", "CD8- T (CD3+CD8-)", "Cytotoxic (CD8+GzmB+)"]):
        plot_density_map(slide, ph, ax=ax)
    plt.tight_layout()
    plt.show()

## 10. Batch normalization and clustering
Unsupervised grouping of cells by their marker intensities, as a complement to the rule-based phenotypes.
- **Markers**: log1p of `sig_<marker>` for the markers that passed QC (flagged channels are left out).
- **Cells**: only cells positive for at least one of those markers (Section 8 thresholds). Most cells in this panel (tumour, stroma) are negative for everything and would only add noise; they are labelled `negative`.
- **Batch correction**: ComBat by `staining_batch` from `metadata/samples.csv`, only when there are at least two batches. The paper estimates ComBat on a control tissue shared by all batches; here it is fitted on all cells, the simplest version.
- **Clustering**: Leiden (resolution 0.6, as in the paper) on a subsample of up to 50,000 cells; the remaining cells get the cluster of their nearest neighbours in marker space. UMAP for display.

In [ ]:
try:
    import scanpy as sc
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scanpy", "igraph", "leidenalg"], check=True)
    import scanpy as sc
import anndata as ad
from sklearn.neighbors import KNeighborsClassifier
warnings.filterwarnings("ignore", module="scanpy|anndata|scipy")

CLUSTER_MARKERS = [m for m in MARKERS if m not in CHANNEL_FLAGS]
LEIDEN_RESOLUTION = 0.6
MAX_CELLS_CLUSTER = 50_000
print("Markers used for clustering:", CLUSTER_MARKERS,
      f"(left out after QC: {list(CHANNEL_FLAGS)})" if CHANNEL_FLAGS else "")

pos_any = cells[[f"{m}_pos" for m in CLUSTER_MARKERS]].any(axis=1).to_numpy()
sel_cells = cells[pos_any]
print(f"Clustering {pos_any.sum():,} cells positive for ≥ 1 marker ({pos_any.mean():.1%} of all cells)")
if pos_any.sum() < 50:
    raise ValueError("Fewer than 50 marker-positive cells: too few to cluster")
X = np.log1p(sel_cells[[f"sig_{m}" for m in CLUSTER_MARKERS]].clip(lower=0).to_numpy(dtype=np.float32))
batch_of = samples.set_index("slide")["staining_batch"].fillna("").astype(str).str.strip()
batch = sel_cells["slide"].map(batch_of).fillna("").replace("", "batch_1").astype(str).to_numpy()
n_batches = len(set(batch))
if n_batches >= 2:
    tmp = ad.AnnData(X.copy(), obs=pd.DataFrame({"batch": pd.Categorical(batch)},
                                                index=np.arange(len(X)).astype(str)))
    sc.pp.combat(tmp, key="batch")
    X = tmp.X.astype(np.float32)
    print(f"ComBat applied over {n_batches} staining batches")
else:
    print("One staining batch only — no batch correction (fill 'staining_batch' in samples.csv to use it)")

Xz = (X - X.mean(0)) / (X.std(0) + 1e-6)
rng = np.random.default_rng(0)
sub = np.sort(rng.choice(len(Xz), min(MAX_CELLS_CLUSTER, len(Xz)), replace=False))
adata = ad.AnnData(Xz[sub], obs=pd.DataFrame({"slide": sel_cells["slide"].to_numpy()[sub], "batch": batch[sub]},
                                             index=sub.astype(str)))
adata.var_names = CLUSTER_MARKERS
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X", random_state=0)
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, flavor="igraph", n_iterations=2,
             key_added="cluster", random_state=0)
sc.tl.umap(adata, random_state=0)

knn = KNeighborsClassifier(n_neighbors=15).fit(Xz[sub], adata.obs["cluster"].to_numpy())
labels_pos = knn.predict(Xz) if len(sub) < len(Xz) else adata.obs["cluster"].to_numpy()
cl_names = sorted(set(labels_pos), key=int)
cells["cluster"] = "negative"
cells.loc[pos_any, "cluster"] = labels_pos
cells["cluster"] = pd.Categorical(cells["cluster"], categories=[*cl_names, "negative"])
cells.to_parquet(OUT_DIR / "cells_phenotyped.parquet")
print(f"{len(cl_names)} clusters among marker-positive cells")

In [ ]:
profile = pd.DataFrame(Xz, columns=CLUSTER_MARKERS).groupby(labels_pos).mean().reindex(cl_names)
size = pd.Series(labels_pos).value_counts().reindex(cl_names)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, max(3, 0.4 * len(profile) + 1.5)),
                               gridspec_kw={"width_ratios": [1, 1.2]})
im = ax1.imshow(profile.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax1.set_xticks(range(len(CLUSTER_MARKERS)), CLUSTER_MARKERS)
ax1.set_yticks(range(len(profile)), [f"{c}  (n={n:,})" for c, n in zip(profile.index, size)])
ax1.set_title("Mean marker level per cluster (z-score)", fontsize=10)
plt.colorbar(im, ax=ax1, shrink=0.8)
um = adata.obsm["X_umap"]
for c in adata.obs["cluster"].cat.categories:
    sel = (adata.obs["cluster"] == c).to_numpy()
    ax2.scatter(um[sel, 0], um[sel, 1], s=2, label=c, rasterized=True)
    ax2.annotate(c, um[sel].mean(0), fontsize=9, weight="bold", ha="center")
ax2.set_title(f"UMAP of {len(sub):,} cells, coloured by Leiden cluster", fontsize=10)
ax2.set_xticks([]); ax2.set_yticks([])
plt.tight_layout()
plt.show()

# Cluster composition per slide (rows sum to 100%) and, for reference, the rule-based phenotypes per cluster
display((pd.crosstab(cells["slide"], cells["cluster"], normalize="index") * 100).round(1))
display((cells.groupby("cluster", observed=True)[list(PHENOTYPES)].mean() * 100).round(1)
        .rename(columns=lambda c: f"% {c}"))
if n_batches >= 2:
    print("Batch mixing: share of each batch in each cluster (similar columns = well mixed)")
    display((pd.crosstab(labels_pos, batch, normalize="index") * 100).round(1).reindex(cl_names)
            .rename_axis(index="cluster", columns="batch"))